<a href="https://colab.research.google.com/github/richayanamandra/GenAI-Experiments/blob/main/genAI_lab4_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Load the Dataset
text_data = """
artificial intelligence is transforming modern society.
it is used in healthcare finance education and transportation.
machine learning allows systems to improve automatically with experience.
data plays a critical role in training intelligent systems.
large datasets help models learn complex patterns.
deep learning uses multi layer neural networks.
neural networks are inspired by biological neurons.
each neuron processes input and produces an output.
training a neural network requires optimization techniques.
gradient descent minimizes the loss function.
natural language processing helps computers understand human language.
text generation is a key task in nlp.
language models predict the next word or character.
recurrent neural networks handle sequential data.
lstm and gru models address long term dependency problems.
however rnn based models are slow for long sequences.
transformer models changed the field of nlp.
they rely on self attention mechanisms.
attention allows the model to focus on relevant context.
transformers process data in parallel.
this makes training faster and more efficient.
modern language models are based on transformers.
education is being improved using artificial intelligence.
intelligent tutoring systems personalize learning.
automated grading saves time for teachers.
online education platforms use recommendation systems.
technology enhances the quality of learning experiences.
ethical considerations are important in artificial intelligence.
fairness transparency and accountability must be ensured.
ai systems should be designed responsibly.
data privacy and security are major concerns.
researchers continue to improve ai safety.
text generation models can create stories poems and articles.
they are used in chatbots virtual assistants and content creation.
generated text should be meaningful and coherent.
evaluation of text generation is challenging.
human judgement is often required.
continuous learning is essential in the field of ai.
research and innovation drive technological progress.
students should build strong foundations in mathematics.
programming skills are important for ai engineers.
practical experimentation enhances understanding.
"""
print("Libraries imported and dataset loaded.")

Libraries imported and dataset loaded.


In [ ]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text_data])
total_words = len(tokenizer.word_index) + 1

# Create input sequences (n-grams)
input_sequences = []
for line in text_data.strip().split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Split into predictors and labels
X, labels = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(labels, num_classes=total_words)

print(f"Total Words in Vocab: {total_words}")
print(f"Max Sequence Length: {max_sequence_len}")
print("Data preprocessed and ready for the model.")

Total Words in Vocab: 195
Max Sequence Length: 10
Data preprocessed and ready for the model.


In [ ]:
# Design the LSTM Architecture
model = Sequential()
model.add(Embedding(input_dim=total_words, output_dim=64, input_length=max_sequence_len-1))
model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Dense(total_words, activation='softmax'))

# Compile the model
optimizer = Adam(learning_rate=0.005)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Train the Model
print("Starting LSTM training...")
model.fit(X, y, epochs=120, verbose=1, batch_size=16)
print("Training complete!")

Starting LSTM training...
Epoch 1/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.0151 - loss: 5.2710
Epoch 2/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0548 - loss: 5.1503
Epoch 3/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0414 - loss: 5.0359
Epoch 4/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0296 - loss: 4.9154
Epoch 5/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0308 - loss: 4.6604
Epoch 6/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0392 - loss: 4.5646
Epoch 7/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0398 - loss: 4.2790
Epoch 8/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1031 - loss: 4.0095
Epoch 9/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1446 - loss: 3.5511
Epoch 10/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1560 - loss: 3.1793
Epoch 11/120
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3261 - loss: 2.6404
Epoch 12/120
16/16 ━━━━━━━━━━━━━━━━

In [ ]:
def generate_text(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        # Convert seed to sequence and pad
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        # Predict
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        # Map back to word
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

words_to_generate = 12
seeds = [
    "artificial intelligence",
    "data plays",
    "text generation models",
    "ethical considerations",
    "machine learning",
    "diameter"
]

for seed in seeds:
    generated = generate_text(seed, words_to_generate, model, max_sequence_len)
    print(f"Seed: '{seed}'\nOutput: '{generated}'\n")

Seed: 'artificial intelligence'
Output: 'artificial intelligence is transforming modern society and transportation creation experience problems problems dependency problems'

Seed: 'data plays'
Output: 'data plays a critical role in training intelligent systems and transportation output output efficient'

Seed: 'text generation models'
Output: 'text generation models can create stories poems and articles articles efficient problems problems efficient efficient'

Seed: 'ethical considerations'
Output: 'ethical considerations are important in artificial intelligence intelligence output creation experience creation problems language'

Seed: 'machine learning'
Output: 'machine learning allows systems to improve automatically with experience experience dependency problems dependency problems'

Seed: 'diameter'
Output: 'diameter judgement is often required on artificial intelligence intelligence output creation creation dependency'

